In [ ]:
import numpy as np
from math import inf
from scipy.optimize import minimize


In [3]:


# --- 1. Définition des g ---

def g_PG(p, q):
    return -4*p*q + 2*p + 4*q - 2

def g_PD(p, q):
    return -4*p*q + 2*p

def g_KG(p, q):
    return 4*p*q - 4*p - 2*q + 2

def g_KD(p, q):
    return 4*p*q - 2*q

# --- 2. Fonction de coût F(p,q) ---

def F_vec(x):
    p, q = x
    gs = [g_PG(p, q), g_PD(p, q), g_KG(p, q), g_KD(p, q)]
    cost = 0.0
    for g in gs:
        r = max(-g, 0.0)  # regret = max(u(k)-u(x),0) = max(-g,0)
        cost += r**2
    return cost

# --- 3. Gradient de F(p,q) ---

def grad_F_vec(x):
    p, q = x
    
    # valeurs des g
    g_vals = {
        'PG': g_PG(p, q),
        'PD': g_PD(p, q),
        'KG': g_KG(p, q),
        'KD': g_KD(p, q),
    }
    
    # dérivées des g
    dg_dp = {
        'PG': -4*q + 2,
        'PD': -4*q + 2,
        'KG':  4*q - 4,
        'KD':  4*q,
    }
    dg_dq = {
        'PG': -4*p + 4,
        'PD': -4*p,
        'KG':  4*p - 2,
        'KD':  4*p - 2,
    }
    
    dF_dp = 0.0
    dF_dq = 0.0
    
    # pour chaque g, si g < 0 -> contribution 2*g*grad(g)
    for key in g_vals:
        g = g_vals[key]
        if g < 0:
            dF_dp += 2*g * dg_dp[key]
            dF_dq += 2*g * dg_dq[key]
    
    return np.array([dF_dp, dF_dq])



In [ ]:
# --- 4. Appel à BFGS ---

# point de départ (arbitraire, hors contraintes pour l'instant)
x0 = np.array([-10, -0.8])
lambda1 = 1
res = minimize(
    fun=F_vec,
    x0=x0,
    jac=grad_F_vec,
    method='BFGS',
    options={'disp': True}
)

print("Résultat BFGS (sans contraintes) :")
print("p* =", res.x[0])
print("q* =", res.x[1])
print("F(p*,q*) =", res.fun)


Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 15
         Function evaluations: 26
         Gradient evaluations: 26
Résultat BFGS (sans contraintes) :
p* = 0.500000382448112
q* = 0.49999903771932236
F(p*,q*) = 4.2889992292810114e-12


In [ ]:
#Ici l'équilibre est bien une probabilité mais il est possible que non => mettre en place un cout pour avoir des valeurs positives et qui forment une distribution

# F_Valeurs_0_1
# grad_F_Valeurs_0,1
def Constraint_0_1(x):
    p,q=x
    return max(-p,0)**2 + max(-q,0)**2 + max(p-1,0)**2 + max(q-1,0)**2

def grad_Constraint_0_1(x):
    p,q=x
    dp=0
    dq=0
    if p<0:
        dp+=2*p
    if q<0:
        dq+=2*q
    if 1-p<0:
        dp+=2*(p-1)
    if 1-q<0:
        dq+=2*(q-1)
    return dp,dq


In [16]:
# --- 4. Appel à BFGS ---
def Func(x):
    p, q = x
    return F_vec(x) + lambda1 * Constraint_0_1(p, q)

def Grad_Func(x):
    p, q = x
    dFp, dFq = grad_F_vec(x)
    dCp, dCq = grad_Constraint_0_1(p, q)
    return np.array([dFp + lambda1*dCp,
                     dFq + lambda1*dCq])


# point de départ (arbitraire, hors contraintes pour l'instant)
x0 = np.array([-10, -0.8])
lambda1 = 10

res = minimize(
    fun= Func,
    x0=x0,
    jac=Grad_Func,
    method='BFGS',
    options={'disp': True}
)

print("Résultat BFGS (avec contraintes) :")
print("p* =", res.x[0])
print("q* =", res.x[1])
print("F(p*,q*) =", res.fun)


Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 23
         Function evaluations: 33
         Gradient evaluations: 33
Résultat BFGS (avec contraintes) :
p* = 0.5000000435318563
q* = 0.49999964182332685
F(p*,q*) = 5.207421281072968e-13


In [59]:
# Payoffs du Player (à toi de fixer ces 4 valeurs)
# u_P(G,G), u_P(G,D), u_P(D,G), u_P(D,D)
A11 = -0.2   # par ex : tir arrêté à gauche
A12 = 0.9   # but si player G, keeper D
A21 = 0.85   # but si player D, keeper G
A22 = -0.4   # tir arrêté à droite

# Pour simplifier les formules
Delta = A11 - A12 - A21 + A22  # terme commun


# ============================================================
# 2. Fonctions auxiliaires B(q) et C(p)
# ============================================================

def B_q(q: float) -> float:
    """
    B(q) = A12 - A22 + q * (A11 - A12 - A21 + A22)
         = A12 - A22 + q * Delta
    Sert pour les g du Player.
    """
    return A12 - A22 + q * Delta


def C_p(p: float) -> float:
    """
    C(p) = A21 - A22 + p * (A11 - A12 - A21 + A22)
         = A21 - A22 + p * Delta
    Sert pour les g du Keeper.
    """
    return A21 - A22 + p * Delta


# ============================================================
# 3. Fonctions g_i(p,q,k) pour Player et Keeper
# ============================================================

def g_PG(p: float, q: float) -> float:
    """
    g_{Player, G}(p,q) = u_P(p,q) - u_P(G,q)
    """
    return (p - 1.0) * B_q(q)


def g_PD(p: float, q: float) -> float:
    """
    g_{Player, D}(p,q) = u_P(p,q) - u_P(D,q)
    """
    return p * B_q(q)


def g_KG(p: float, q: float) -> float:
    """
    g_{Keeper, G}(p,q) = u_K(p,q) - u_K(p,G)
    """
    return (1.0 - q) * C_p(p)


def g_KD(p: float, q: float) -> float:
    """
    g_{Keeper, D}(p,q) = u_K(p,q) - u_K(p,D)
    """
    return -q * C_p(p)


# ============================================================
# 4. Fonction de coût F(p,q)
# ============================================================

def F_vec(x: np.ndarray) -> float:
    """
    x = [p, q]
    F(p,q) = somme des regrets positifs au carré :
      F = sum_{i,k} max(-g_{i,k}(p,q), 0)^2
    """
    p, q = x
    gs = [
        g_PG(p, q),  # Player, déviation vers G
        g_PD(p, q),  # Player, déviation vers D
        g_KG(p, q),  # Keeper, déviation vers G
        g_KD(p, q),  # Keeper, déviation vers D
    ]
    cost = 0.0
    for g in gs:
        r = max(-g, 0.0)  # regret = max(u(k)-u(x), 0) = max(-g, 0)
        cost += r * r
    return cost


# ============================================================
# 5. Gradient de F(p,q)
# ============================================================

def grad_F_vec(x: np.ndarray) -> np.ndarray:
    """
    Gradient de F par rapport à (p,q).

    Pour chaque g:
      t(g) = max(-g, 0)^2
      si g < 0:  t = g^2  et dt/dx = 2*g * dg/dx
      si g >= 0: t = 0    et dt/dx = 0
    """
    p, q = x

    B = B_q(q)
    C = C_p(p)
    Bp = Delta  # dB/dq
    Cp = Delta  # dC/dp

    # valeurs des g
    g_vals = {
        'PG': g_PG(p, q),
        'PD': g_PD(p, q),
        'KG': g_KG(p, q),
        'KD': g_KD(p, q),
    }

    # dérivées partielles des g
    # Player
    dgPG_dp = B
    dgPG_dq = (p - 1.0) * Bp

    dgPD_dp = B
    dgPD_dq = p * Bp

    # Keeper
    dgKG_dp = (1.0 - q) * Cp
    dgKG_dq = -C

    dgKD_dp = -q * Cp
    dgKD_dq = -C

    dg_dp = {
        'PG': dgPG_dp,
        'PD': dgPD_dp,
        'KG': dgKG_dp,
        'KD': dgKD_dp,
    }
    dg_dq = {
        'PG': dgPG_dq,
        'PD': dgPD_dq,
        'KG': dgKG_dq,
        'KD': dgKD_dq,
    }

    dF_dp = 0.0
    dF_dq = 0.0

    # pour chaque g, si g < 0 -> contribution 2*g*grad(g)
    for key, g in g_vals.items():
        if g < 0.0:
            dF_dp += 2.0 * g * dg_dp[key]
            dF_dq += 2.0 * g * dg_dq[key]

    return np.array([dF_dp, dF_dq])


In [60]:
# --- 4. Appel à BFGS ---

# point de départ (arbitraire, hors contraintes pour l'instant)
x0 = np.array([-10, 0.8])
lambda1 = 0
res = minimize(
    fun=F_vec,
    x0=x0,
    jac=grad_F_vec,
    method='BFGS',
    options={'disp': True}
)

print("Résultat BFGS (sans contraintes) :")
print("p* =", res.x[0])
print("q* =", res.x[1])
print("F(p*,q*) =", res.fun)


Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 24
         Function evaluations: 32
         Gradient evaluations: 32
Résultat BFGS (sans contraintes) :
p* = 0.5319158989638897
q* = 0.5531887449642274
F(p*,q*) = 1.0227674079752683e-11
